# Importando bibliotecas

In [ ]:
import pandas as pd
import plotly.express as px

# Importando dados

In [ ]:
df_metricas = pd.read_excel('../data/raw/Relatório de Produção - Completo - Dados.xlsx')
df_bloco_notas = pd.read_excel('../data/raw/Relatório de Produção - Completo - Bloco de Notas.xlsx')

# Visualizando dados

In [ ]:
display(df_metricas)
display(df_bloco_notas)

# Tratando dados

In [ ]:
df_metricas.columns = df_metricas.iloc[1]
df_metricas = df_metricas.iloc[2:].reset_index(drop=True)
df_metricas.columns.name = None

colunas_float = [
    "Eficiencia Plan",
    "Hora Hora Wht (6to6)",
    "Veloc Stand",
    "Veloc Real",
    "Qtd Teorica Real",
    "Corte de Gota %",
    "Objetivo %",
    "Qtd Objetivo",
    "Empacotado %",
    "Qtd Empacotado",
    "Qtd Rejeicao",
    "Rejeição %",
]

for coluna in colunas_float:
    df_metricas[coluna] = pd.to_numeric(
        df_metricas[coluna]
        .astype("string")
        .str.strip()
        .str.replace("%", "", regex=False)
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )

df_metricas['Prefixo'] = df_metricas['Prefixo'].str.replace(' ', '').copy()

# criando variavel de data e hora para juncao com blocos de notas
df_metricas["Data"] = (
    pd.to_datetime(
        df_metricas["Data Wht (dia)"],
        format="%d/%m/%Y",
        errors="coerce"
    )
    + pd.to_timedelta(
        df_metricas["Hora Hora Wht (6to6)"],
        unit="h"
    )
)

display(df_metricas)


In [ ]:
# tratamentos df_bloco_notas
display(df_bloco_notas.info())

df_bloco_notas = df_bloco_notas.drop(columns=['Unnamed: 1'])

df_bloco_notas.columns = df_bloco_notas.iloc[1]
df_bloco_notas = df_bloco_notas.iloc[2:].reset_index(drop=True)
df_bloco_notas.columns.name = None

display(df_bloco_notas.info())

df_bloco_notas['Prefixo'] = df_bloco_notas['Prefixo'].str.replace(' ', '').copy()

df_bloco_notas.rename(columns={'Hora': 'Data'}, inplace=True)
df_bloco_notas['Data'] = pd.to_datetime(df_bloco_notas['Data'])

display(df_bloco_notas)

# Juntando dados

## Anotações

A base de métricas registra a produção por hora, enquanto a base de notas contém data, hora, minuto e segundo.

A data das duas bases é truncada para a hora em `Data_merge`, preservando o horário exato original em `Data_metrica` e `Data_nota`.

Chaves do merge: `Data_merge`, `Maquina` e `Prefixo`. A cardinalidade esperada é um-para-muitos.

In [ ]:
df_metricas

In [ ]:
df_bloco_notas

In [ ]:
# Garante que ambas as colunas sejam datetime
df_metricas["Data"] = pd.to_datetime(
    df_metricas["Data"],
    dayfirst=True,
    errors="coerce"
)

df_bloco_notas["Data"] = pd.to_datetime(
    df_bloco_notas["Data"],
    dayfirst=True,
    errors="coerce"
)

# Cria a chave horária para o merge
df_metricas["Data_merge"] = df_metricas["Data"].dt.floor("h")
df_bloco_notas["Data_merge"] = df_bloco_notas["Data"].dt.floor("h")

# Padroniza as demais chaves
chaves_texto = ["Maquina", "Prefixo"]

for coluna in chaves_texto:
    df_metricas[coluna] = (
        df_metricas[coluna]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    df_bloco_notas[coluna] = (
        df_bloco_notas[coluna]
        .astype("string")
        .str.strip()
        .str.upper()
    )

# Cria cópias com identificadores para permitir a auditoria do merge
metricas_merge = df_metricas.copy().reset_index(drop=True)
notas_merge = df_bloco_notas.copy().reset_index(drop=True)

metricas_merge["_id_metrica"] = metricas_merge.index
notas_merge["_id_nota"] = notas_merge.index

chaves_merge = ["Data_merge", "Maquina", "Prefixo"]

# Merge: one to many -> uma métrica pode estar associada a várias notas
df_final = metricas_merge.merge(
    notas_merge,
    how="left",
    on=chaves_merge,
    suffixes=("_metrica", "_nota"),
    validate="one_to_many",
    indicator="_status_merge"
)

# Validando a junção dos dados

As verificações abaixo auditam as chaves, a cardinalidade, os registros sem correspondência e a quantidade final de linhas.

In [ ]:
# 1. Verifica valores ausentes nas chaves do merge
resumo_chaves_nulas = pd.DataFrame({
    "metricas": metricas_merge[chaves_merge].isna().sum(),
    "notas": notas_merge[chaves_merge].isna().sum(),
})
display(resumo_chaves_nulas)

assert not metricas_merge[chaves_merge].isna().any().any(), (
    "Existem métricas com alguma chave nula. Verifique Data_merge, Maquina e Prefixo."
)

# 2. Confirma que a base de métricas representa o lado 'um'
metricas_duplicadas = (
    metricas_merge.loc[
        metricas_merge.duplicated(subset=chaves_merge, keep=False)
    ]
    .sort_values(chaves_merge)
)

print(f"Métricas com chave duplicada: {len(metricas_duplicadas)}")
if not metricas_duplicadas.empty:
    display(metricas_duplicadas)

assert metricas_duplicadas.empty, (
    "A combinação Data_merge, Maquina e Prefixo não é única em df_metricas."
)

In [ ]:
# 3. Resume os resultados do left merge
resumo_merge = (
    df_final["_status_merge"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="quantidade_linhas")
)
display(resumo_merge)

metricas_sem_nota = df_final.loc[
    df_final["_status_merge"].eq("left_only")
].copy()

print(
    "Métricas sem nota:",
    metricas_sem_nota["_id_metrica"].nunique(),
)

print('Abaixo, os registros das métricas sem notas: ')
display(metricas_sem_nota)

In [ ]:
# 4. Localiza notas que não encontraram métrica correspondente
notas_auditadas = notas_merge.merge(
    metricas_merge[chaves_merge].drop_duplicates(),
    how="left",
    on=chaves_merge,
    indicator="_tem_metrica",
)

notas_sem_metrica = notas_auditadas.loc[
    notas_auditadas["_tem_metrica"].eq("left_only")
].copy()

print(f"Notas sem métrica correspondente: {len(notas_sem_metrica)}")
if not notas_sem_metrica.empty:
    display(notas_sem_metrica)

In [ ]:
# 5. Calcula quantas linhas o resultado deveria possuir
notas_por_chave = (
    notas_merge.groupby(chaves_merge, dropna=False)
    .size()
    .rename("quantidade_notas")
    .reset_index()
)

validacao_quantidade = metricas_merge[
    ["_id_metrica"] + chaves_merge
].merge(
    notas_por_chave,
    how="left",
    on=chaves_merge,
)

validacao_quantidade["quantidade_notas"] = (
    validacao_quantidade["quantidade_notas"]
    .fillna(0)
    .astype(int)
)

linhas_esperadas = int(
    validacao_quantidade["quantidade_notas"]
    .clip(lower=1)
    .sum()
)
linhas_encontradas = len(df_final)

print(f"Linhas esperadas: {linhas_esperadas}")
print(f"Linhas encontradas: {linhas_encontradas}")

assert linhas_encontradas == linhas_esperadas, (
    "A quantidade de linhas resultante do merge é diferente da esperada."
)

assert df_final["_id_metrica"].nunique() == len(metricas_merge), (
    "Uma ou mais métricas desapareceram durante o left merge."
)

In [ ]:
# 6. Confirma que as notas associadas pertencem à mesma hora da métrica
linhas_com_nota = df_final["_status_merge"].eq("both")

horarios_consistentes = (
    df_final.loc[linhas_com_nota, "Data_metrica"].dt.floor("h")
    .eq(df_final.loc[linhas_com_nota, "Data_merge"])
    & df_final.loc[linhas_com_nota, "Data_nota"].dt.floor("h")
    .eq(df_final.loc[linhas_com_nota, "Data_merge"])
)

assert horarios_consistentes.all(), (
    "Há notas associadas a uma hora diferente da hora da métrica."
)

# Como a chave das métricas é única, cada nota encontrada deve aparecer uma vez
ocorrencias_por_nota = (
    df_final.loc[linhas_com_nota, "_id_nota"]
    .value_counts()
)
assert ocorrencias_por_nota.le(1).all(), (
    "Uma mesma nota foi associada a mais de uma métrica."
)

resumo_validacao = pd.Series({
    "metricas_origem": len(metricas_merge),
    "notas_origem": len(notas_merge),
    "metricas_com_nota": df_final.loc[linhas_com_nota, "_id_metrica"].nunique(),
    "metricas_sem_nota": metricas_sem_nota["_id_metrica"].nunique(),
    "notas_associadas": df_final.loc[linhas_com_nota, "_id_nota"].nunique(),
    "notas_sem_metrica": len(notas_sem_metrica),
    "linhas_resultado": len(df_final),
}, name="quantidade")

display(resumo_validacao.to_frame())
print("Validações estruturais concluídas com sucesso.")

In [ ]:
# Remove somente as colunas auxiliares após concluir as validações
df_final = df_final.drop(
    columns=["Data_merge", "_id_metrica", "_id_nota", "_status_merge"],
    errors="ignore",
)

In [ ]:
display(df_final)

In [ ]:
df_final.info()


# Análises

## Anotações 

Insights:
- Variável "Qtd Teorica Real" tem comportamento similar ao "Veloc Real" por OP hora a hora
    - Possivelmente as duas possuem correlação alta

- Empacotado < Rejeitado
    - Notado que há OPs com qtd rejeitada maior que qtd empacotada, não faz sentido

Futuras análises: 
- Calcular um valor que represente a performance de uma OP
    - Mostrar as OPs com menores performances
    - Ver variáveis Empacotado, Objetivo e Rejeitado de hora em hora
        - Fazer um contador de quantas vezes OP ficou abaixo do objetivo?
        - Fazer alguma coisa com rejeitado também!

## Copiando a base para análises

In [ ]:
df = df_final.copy()
df

## Informações das variáveis

In [ ]:
df.info()

## Correlação entre variáveis

In [ ]:
# vairaveis numericas
df_numerico = df.select_dtypes(include="number")

# removendo variaveis categoricas ou não-variáveis
df_numerico = df_numerico.drop(columns={
    'Corte de Gota %'
})

# calculo matriz de correlacao
matriz_correlacao = df_numerico.corr(method="pearson")

# plotagem da matriz
fig = px.imshow(
    matriz_correlacao,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Matriz de Correlação das Variáveis"
)

fig.update_layout(
    xaxis_title="Variáveis",
    yaxis_title="Variáveis",
    width=1000,
    height=800
)

fig.show()

## Variação de velocidade real por OP hora a hora

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_metrica', inplace=True)

fig = px.line(
    df_analise,
    x='Data_metrica',
    y='Veloc Real',
    title='Variação de "Veloc Real" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    labels={
        'Data_metrica': 'Data'
    }
)

fig.show()

## Variação da quantidade empacotada por OP hora a hora

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_metrica', inplace=True)

fig = px.line(
    df_analise,
    x='Data_metrica',
    y='Empacotado %',
    title='Variação de "Empacotado" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    labels={
        'Data_metrica': 'Data'
    },
    hover_data=[
        'Qtd Empacotado',
        'Rejeição %',
        'Qtd Rejeicao',
        'Objetivo %',
        'Qtd Objetivo'
    ]
)

fig.show()

## Variação de "Rejeição" por OP hora a hora


In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_metrica', inplace=True)

fig = px.line(
    df_analise,
    x='Data_metrica',
    y='Rejeição %',
    title='Variação de "Rejeição" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    labels={
        'Data_metrica': 'Data'
    },
    hover_data=[
        'Qtd Rejeicao',
        'Empacotado %',
        'Qtd Empacotado',
        'Objetivo %',
        'Qtd Objetivo'
    ]
)

fig.show()

In [ ]:
print('Casos onde "Qtd Rejeicao" > "Qtd Empacotado": ')
df_analise.query('`Qtd Rejeicao` > `Qtd Empacotado`')

## Variação de "Qtd Teorica Real" por OP hora a hora

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_metrica', inplace=True)

fig = px.line(
    df_analise,
    x='Data_metrica',
    y='Qtd Teorica Real',
    title='Variação de "Qtd Teorica Real" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    labels={
        'Data_metrica': 'Data'
    },
    hover_data=[
        'Empacotado %',
        'Qtd Empacotado',
        'Rejeição %',
        'Qtd Rejeicao',
        'Objetivo %',
        'Qtd Objetivo'
    ]
)

fig.show()

## Variação do corte de gota por OP hora a hora

In [ ]:
df_analise = df.copy()
df_analise.sort_values(by='Data_metrica', inplace=True)

fig = px.line(
    df_analise,
    x='Data_metrica',
    y='Corte de Gota %',
    title='Variação de "Corte de Gota %" por OP hora a hora',
    color='OP Vertech',
    markers=True,
    labels={
        'Data_metrica': 'Data'
    }
)

fig.show()